# HEG-GRPO control panel

Run the cells top to bottom, one phase at a time. Training is launched into a background
`tmux` session, so closing this tab, the browser or the SSH tunnel never stops a run.
Re-running a launch cell while a run is active is refused, so nothing gets double-started.

| Phase | What | Cost |
|---|---|---|
| 0 | Settings + environment check | $0 |
| 1 | Pilot (calibrate + 2 x 30 steps + 10-step SELF check) + pilot report | ~$1-2 |
| 2 | Decide steps / lr from the report | $0 |
| 3 | Main queue (unattended, auto-shutdown) | per pilot report |
| 4 | Results table + plots | $0 |

## 0. Settings (edit `S3_URI` once)

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
from IPython.display import Image, display

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))
ART = Path(os.environ.get('ARTIFACT_ROOT', Path.home() / 'artifacts'))

S3_URI = 's3://CHANGE-ME-your-bucket-name/heg-grpo'   # your bucket from README A5
USD_PER_HOUR = 1.006                                  # g5.xlarge on-demand, us-east-1
BUDGET_USD = 200


def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=REPO)
    print((r.stdout + r.stderr).rstrip())
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.returncode


def tmux_running(name='train'):
    return subprocess.run(['tmux', 'has-session', '-t', name], capture_output=True).returncode == 0


def launch(queue, extra=''):
    if 'CHANGE-ME' in S3_URI:
        raise ValueError('Set S3_URI in the settings cell first.')
    if tmux_running():
        raise RuntimeError("A run is already active (tmux session 'train'). Check status below.")
    (ART / 'logs').mkdir(parents=True, exist_ok=True)
    qlog = ART / 'logs' / f'queue_{Path(queue).stem}.log'
    inner = (f'bash scripts/run_queue.sh {queue} --s3 {S3_URI} --usd-per-hour {USD_PER_HOUR} {extra} '
             f'2>&1 | tee -a {qlog}')
    subprocess.run(['tmux', 'new-session', '-d', '-s', 'train', '-c', str(REPO), inner], check=True)
    print(f'Launched in tmux session "train". Queue log: {qlog}')


def status(n=15):
    print('tmux session "train":', 'RUNNING' if tmux_running() else 'not running')
    for q in sorted((ART / 'logs').glob('queue_*.log')):
        print(f'\n--- {q.name} (last {n} lines)')
        print(''.join(q.read_text().splitlines(keepends=True)[-n:]))
    for log in sorted((ART / 'runs').glob('*/*/seed*/stdout.log')):
        lines = log.read_text().strip().splitlines()
        print(f'{str(log.parent.relative_to(ART / "runs")):38s} {lines[-1][:170] if lines else ""}')


print('repo:', REPO)
print('artifacts:', ART)

### Environment check: must show the A10G, `bf16= True`, an IAM role ARN, the bucket, and passing tests

In [ ]:
sh('nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv')
sh(f"{sys.executable} -c \"import torch; print(torch.__version__, torch.cuda.get_device_name(0), 'bf16=', torch.cuda.is_bf16_supported())\"")
sh('aws sts get-caller-identity --query Arn --output text')
bucket = S3_URI.split('/')[2]
sh(f'aws s3 ls s3://{bucket} > /dev/null && echo "S3 bucket reachable: {bucket}"')
sh('df -h ~ | tail -1')
sh(f'{sys.executable} -m pytest -q tests -p no:cacheprovider 2>&1 | tail -1')

## 1. Pilot (~1-1.5 h, ~$1-2)

`calibrate` (chooses MEG's mode partitioner on base-model rollouts), two 30-step `heg_grpo` runs at
lr 1e-6 and 1e-5 (each times a 16-problem eval probe), and a 10-step `o_self` run. No final evals.
Launch once, then re-run the status cell every few minutes (curves: open `monitor.ipynb`).
Safety: the machine powers itself off 4 h after launch even if you walk away (the pilot needs
~1.5 h). Need longer? In a terminal: `sudo shutdown -c`.

In [ ]:
launch('queues/pilot.txt', '--max-hours 4')

In [ ]:
status()

### Pilot report (run after every pilot line shows DONE in the status output)

In [ ]:
PLANNED_STEPS = 800      # what you would like; the report says whether 18 core (+6 extra) runs fit
sh(f"cat {ART}/calibration/0.5B/calibration.json | head -20")
for run in ('heg_grpo__pilot_lr1e-6', 'heg_grpo__pilot_lr1e-5', 'o_self__pilot_self'):
    sh(f'{sys.executable} pilot_report.py --run {run} --usd_per_hour {USD_PER_HOUR} '
       f'--budget {BUDGET_USD} --steps {PLANNED_STEPS}')

## 2. Decide (paste the reports to Claude if unsure)

- **LR**: the one whose KL to base clearly rises (likely 1e-5). If neither moves: stop.
- **STEPS**: the largest step count the report says fits in 85% of the budget.
- **MAX_HOURS**: expected total queue hours + ~10%. Dead-man switch: the instance powers off
  at that point no matter what.
- Write the chosen lr, steps, levels and the calibrated partitioner into the `[pilot]` slots of
  `PREREGISTRATION.md` **before** launching the main queue.

## 3. Main queue (unattended)

When the queue finishes, the instance shuts itself down (compute billing stops). If you come
back and this page doesn't load, that is why: start the instance in the EC2 console, reconnect
(new public IP), and run the results cell. After any interruption, re-run the launch cell:
finished runs are skipped and the active one resumes from its last checkpoint.

In [ ]:
LR, STEPS, MAX_HOURS = 1e-5, 400, 100      # <- set from the pilot reports
QUEUE = 'queues/main_0.5B.txt'              # or 'queues/main_1.5B.txt'
launch(QUEUE, f'--train-args "--steps {STEPS} --lr {LR}" --max-hours {MAX_HOURS} --shutdown-when-done')

In [ ]:
status()

### Extras (only if the pilot report said "core + extras ... fits")

Random-partition control and the BBG-style baseline. Launch after the main queue has finished
(start the instance again if it shut itself down), with the same `LR` / `STEPS`.

In [ ]:
launch('queues/extras_0.5B.txt', f'--train-args "--steps {STEPS} --lr {LR}" --max-hours {MAX_HOURS} --shutdown-when-done')

## 4. Results (after the queue is done)

In [ ]:
for size in ('0.5B',):          # add '1.5B' if you ran that queue
    for bench in ('math500', 'gsm8k'):
        rc = sh(f'{sys.executable} analyze.py --model_size {size} --benchmark {bench}')
        png = ART / 'analysis' / f'{bench}_{size}.png'
        if rc == 0 and png.exists():
            display(Image(str(png)))
sh(f'aws s3 sync {ART} {S3_URI} --only-show-errors && echo "synced to S3"')

## 5. Human evaluation (PREREGISTRATION.md "Human evaluation"; ~4-5 h per annotator, no GPU needed)

Builds blinded spreadsheets for two annotators. Do this **before terminating** the instance: it needs the
saved base/vanilla completions. Download `human_eval/annotator_A` and `annotator_B` (right-click -> Download,
or from S3), give each to one person together with `HUMAN_EVAL_GUIDE.md`, and keep `key.json` to yourself.

In [ ]:
sh(f'{sys.executable} scripts/human_eval.py sheets')
sh(f'aws s3 sync {ART}/human_eval {S3_URI}/human_eval --only-show-errors && echo "sheets synced to S3"')

When both filled-in CSVs are back in `~/artifacts/human_eval/annotator_A|B/` (same file names), score them.
This also works on the laptop with the downloaded artifacts: `python scripts/human_eval.py score --artifact_root ...`.

In [ ]:
sh(f'{sys.executable} scripts/human_eval.py score')

## Emergency controls (JupyterLab: File -> New -> Terminal)

| What | Command |
|---|---|
| Watch the live run | `tmux attach -t train`  (leave with Ctrl-b then d) |
| Stop the queue now | `tmux kill-session -t train`  (resume later: re-run the launch cell) |
| Cancel the safety shutdown | `sudo shutdown -c` |
| Power off now | `sudo shutdown -h now` |
| GPU usage | `watch -n 2 nvidia-smi` |